# Quickstart L2: global semantics

**L2** adds *global semantics* on top of the L1 core of [Quickstart L1: local semantics](quickstart_l1.ipynb). A *logical mesh* is the whole mesh, with global indices: its tables, regions, geometry and identity; its fields carry it. A *global program* is written for the whole mesh, without ranks, halos or communication: run directly, it is the reference; *lowered* onto a distribution of the mesh, it runs as a local program on every rank, with the same results, bitwise. This quickstart walks through it with the example of L1, a section per topic, each with a link to the in-depth notebook; all the notebooks are listed in [0. Overview](00_overview.ipynb).

The example: heat diffusion along the row of 4 cells and 5 edges of L1, now a logical mesh, with two regions, `West` (the cell `c0`) and `East` (the cell `c3`).

```
          e0      e1      e2      e3      e4
outside   |  c0   |  c1   |  c2   |  c3   |   outside
t_out=10  |  10   |  20   |  40   |  30   |   t_out=10
             West                    East
```

Contents: schemas and locations · the logical mesh · mesh fields · L1 operators, unchanged · programs · distributions · the lowering · backends and the compile key · L1 next to L2 · cheat sheet.

The in-depth notebooks: [12. Logical meshes](12_logical_meshes.ipynb) · [13. Global programs](13_global_programs.ipynb), and, for the L1 side of the last section, [10. Local programs](10_local_programs.ipynb).

## Schemas and locations

A schema is a brand for the locations of its meshes, declared by subclassing `Schema`: `Face[Row]`, `Edge[Row]` and `Node[Row]` are its families, UGRID's 2-, 1- and 0-dimensional locations, each an ordinary L1 `Location`. Here `Cell` names `Face[Row]` and `Edge` names `Edge[Row]`; the family itself is written `concepts.Edge`, so that the L1 code below, which says `Edge`, runs unchanged.

In depth: [12. Logical meshes](12_logical_meshes.ipynb).

In [1]:
import contextlib
import io

import concepts
from concepts import Face, Location, Schema, SchemaLocation, location_of


class Row(Schema): ...  # the schema: a brand for the locations of its meshes


Cell = Face[Row]  # the cells of the row: its faces
Edge = concepts.Edge[Row]  # its edges: the family `concepts.Edge`, of the schema Row
print("Cell:", Cell.__name__, "| Edge:", Edge.__name__, "| schema:", Edge.schema.__name__)
print(
    "Locations:",
    issubclass(Cell, Location) and issubclass(Edge, Location),
    "| SchemaLocations:",
    issubclass(Edge, SchemaLocation),
    "| location_of(concepts.Edge, Row) is Edge:",
    location_of(concepts.Edge, Row) is Edge,
)
try:
    Row()
except TypeError as error:
    print("TypeError:", error)

Cell: Face[Row] | Edge: Edge[Row] | schema: Row
Locations: True | SchemaLocations: True | location_of(concepts.Edge, Row) is Edge: True
TypeError: A schema is a type: it is never instantiated.


## The logical mesh

`LogicalMesh(Row, tables, extents=..., regions=..., geometry=...)` is the whole mesh, built once, checked and immutable: its tables, the extents of its locations, its regions (global index sets of `Region[D]` classes), its geometry (named fields) and its identity, the `mesh_id`. Its `signature`, the schema, the relations and the regions, is the compile key: meshes with the same signature share their compiled operators (see "Backends and the compile key").

In depth: [12. Logical meshes](12_logical_meshes.ipynb).

In [2]:
from concepts import MISSING_VALUE, LogicalMesh, Region, Relation
from implementation import NumPyArrayField, format_domain, neighbor_table


class C2E(Relation[Cell, Edge], max_neighbors=2): ...  # [left edge, right edge]


class E2C(Relation[Edge, Cell], max_neighbors=2, partial=True): ...  # [left cell, right cell]


class West(Region[Cell]): ...  # the cell at the west end, c0


class East(Region[Cell]): ...  # the cell at the east end, c3


M = MISSING_VALUE


def row_tables(n):  # a row of n cells and n + 1 edges, edge e between the cells e - 1 and e
    return {
        C2E: neighbor_table(C2E, [[c, c + 1] for c in range(n)]),
        E2C: neighbor_table(
            E2C, [[e - 1 if e > 0 else M, e if e < n else M] for e in range(n + 1)]
        ),
    }


length = NumPyArrayField({Cell: range(4)}, [0.5, 0.5, 0.5, 0.5])  # the length of each cell
row = LogicalMesh(
    Row,
    row_tables(4),
    extents={Cell: 4, Edge: 5},
    regions={West: (0,), East: (3,)},
    geometry={"length": length},
)
print(row)  # Row-0, its `mesh_id`: the first logical mesh of this kernel
print("E2C =", row[E2C].data.tolist())  # e0: [M, c0], ..., e4: [c3, M]
print("cells:", format_domain(row.domain(Cell)), "| East:", format_domain(row.region(East)))
schema, relations, regions = row.signature  # the compile key: classes, sorted by name
print("signature:", schema.__name__, [r.__name__ for r in relations], [r.__name__ for r in regions])

LogicalMesh('Row-0')
E2C = [[-2, 0], [0, 1], [1, 2], [2, 3], [3, -2]]
cells: Face[Row][0:4] | East: Face[Row]{3}
signature: Row ['C2E', 'E2C'] ['East', 'West']


## Mesh fields

`row.field`, `row.zeros` and `row.full` give `MeshField`s on the whole of some dimensions of the mesh: they carry it, and read through its tables without a `with` block. Fields of different logical meshes do not combine (a `ValueError`). The local fields of signs have a neighbor dimension, which has no extent in the mesh: `mesh_field(row, domain, data)` builds them.

In depth: [12. Logical meshes](12_logical_meshes.ipynb).

In [3]:
from concepts import MeshField, Neighbor
from implementation import mesh_field, sum_over

t = row.field(Cell, data=[10.0, 20.0, 40.0, 30.0])  # a temperature per cell
t_out = 10.0  # the temperature outside the row
print("t =", t, "| a MeshField:", isinstance(t, MeshField), "| t.mesh is row:", t.mesh is row)
sides = t(E2C.boundary(t_out))  # no `with` block: the tables of t's mesh
# e0: 10 + 10 = 20, e1: 10 + 20 = 30, e2: 20 + 40 = 60, e3: 40 + 30 = 70, e4: 30 + 10 = 40
print("sum_over =", sum_over(sides, axis=Neighbor[E2C]))
print("heat     =", row.geometry["length"] * t)  # 0.5 * t
print("zeros    =", row.zeros(Edge))
print("in West  =", row.full(Cell, fill_value=t_out)[row.region(West)])


def signs(mesh):  # the local fields of signs for "right - left", on the edges and on the cells
    n = mesh.extents[Cell]
    edge_domain = {Edge: range(n + 1), Neighbor[E2C]: range(2)}
    cell_domain = {Cell: range(n), Neighbor[C2E]: range(2)}
    return (
        mesh_field(mesh, edge_domain, [[-1.0, 1.0]] * (n + 1)),
        mesh_field(mesh, cell_domain, [[-1.0, 1.0]] * n),
    )


edge_sign, cell_sign = signs(row)
other = LogicalMesh(Row, row_tables(4), extents={Cell: 4, Edge: 5})  # the same tables: Row-1
try:
    t + other.field(Cell, data=[1.0, 1.0, 1.0, 1.0])
except ValueError as error:
    print("ValueError:", error)

t = Field(Face[Row][0:4]: [10.0, 20.0, 40.0, 30.0]) | a MeshField: True | t.mesh is row: True
sum_over = Field(Edge[Row][0:5]: [20.0, 30.0, 60.0, 70.0, 40.0])
heat     = Field(Face[Row][0:4]: [5.0, 10.0, 20.0, 15.0])
zeros    = Field(Edge[Row][0:5]: [0.0, 0.0, 0.0, 0.0, 0.0])
in West  = Field(Face[Row]{0}: [10.0])
ValueError: The fields are on different logical meshes, 'Row-0' and 'Row-1'.


## L1 operators, unchanged

The field operators of [Quickstart L1](quickstart_l1.ipynb), copied verbatim: L2 adds no new kind of operator. Called with mesh fields, they run in the mesh of their arguments, without a `with` block, on any backend; their results are fields of that mesh (except on the lazy backends, whose `DeferredField`s carry no mesh).

In depth: [12. Logical meshes](12_logical_meshes.ipynb).

In [4]:
from concepts import Dims, Field
from implementation import eager, field_operator


@field_operator()
def flux(
    t: Field[Dims[Cell], float], t_out: float, sign: Field[Dims[Edge, Neighbor[E2C]], float]
) -> Field[Dims[Edge], float]:
    return sum_over(sign * t(E2C.boundary(t_out)), axis=Neighbor[E2C])  # right cell - left cell


@field_operator()
def divergence(
    f: Field[Dims[Edge], float], sign: Field[Dims[Cell, Neighbor[C2E]], float]
) -> Field[Dims[Cell], float]:
    return sum_over(sign * f(C2E), axis=Neighbor[C2E])  # right edge - left edge


@field_operator()
def heat_step(
    t: Field[Dims[Cell], float],
    t_out: float,
    k: float,
    edge_sign: Field[Dims[Edge, Neighbor[E2C]], float],
    cell_sign: Field[Dims[Cell, Neighbor[C2E]], float],
) -> Field[Dims[Cell], float]:
    return t + k * divergence(flux(t, t_out, edge_sign), cell_sign)


k = 0.25
# as in L1: fluxes 0, 10, 20, -10, -20; c0: 10 + 10 / 4 = 12.5, c1: 20 + 10 / 4 = 22.5,
# c2: 40 - 30 / 4 = 32.5, c3: 30 - 10 / 4 = 27.5
t1 = heat_step(t, t_out, k, edge_sign, cell_sign)  # no `with` block either
print("heat_step =", t1, "| a MeshField:", isinstance(t1, MeshField), "| on row:", t1.mesh is row)
with contextlib.redirect_stdout(io.StringIO()):  # silence the backend
    on_eager = eager(heat_step)(t, t_out, k, edge_sign, cell_sign)
print("on eager  =", on_eager.data.tolist(), "| on row:", on_eager.mesh is row)

heat_step = Field(Face[Row][0:4]: [12.5, 22.5, 32.5, 27.5]) | a MeshField: True | on row: True
on eager  = [12.5, 22.5, 32.5, 27.5] | on row: True


## Programs

`@program()` makes a global program of a Python function, after checking it with `PROGRAM_RULES`: it calls operators in loops and branches, and requests regions, `op[region(East)](...)`. `sum_over(t, axis=Cell)`, over a `Location`, is a global sum: the values added in the order of the global indices, the same for any decomposition. What depends on the decomposition, e.g. `exchange` or `Owned`, is rejected.

In depth: [13. Global programs](13_global_programs.ipynb).

In [5]:
from concepts import region
from implementation import exchange, program

type CellField = Field[Dims[Cell], float]
type EdgeSigns = Field[Dims[Edge, Neighbor[E2C]], float]
type CellSigns = Field[Dims[Cell, Neighbor[C2E]], float]


@program()
def simulate(
    t: CellField, t_out: float, k: float, edge_sign: EdgeSigns, cell_sign: CellSigns, steps: int
) -> tuple[CellField, float, CellField]:
    for _ in range(steps):
        t = heat_step(t, t_out, k, edge_sign, cell_sign)
    next_east = heat_step[region(East)](t, t_out, k, edge_sign, cell_sign)  # one more, on East
    return t, sum_over(t, axis=Cell), next_east  # sum_over a `Location`: a global sum


args = (t, t_out, k, edge_sign, cell_sign, 2)
direct = simulate(*args)
# step 2, from 12.5, 22.5, 32.5, 27.5 (fluxes 2.5, 10, 10, -5, -17.5): 12.5 + 7.5 / 4 = 14.375,
# 22.5 + 0 / 4 = 22.5, 32.5 - 15 / 4 = 28.75, 27.5 - 12.5 / 4 = 24.375; their sum: 90;
# one more step on East: 24.375 + (-14.375 + 4.375) / 4 = 21.875
print("direct:", direct)  # results give consecutive index sets as ranges: East, {3}, as [3:4]
print("operators:", [op.__name__ for op in simulate.operators])  # those the body calls


def with_exchange(t: CellField) -> CellField:
    return exchange(t)


try:
    program()(with_exchange)  # rejected by `PROGRAM_RULES`: a collective, `exchange`
except TypeError as error:
    print("TypeError:", error)

direct: (Field(Face[Row][0:4]: [14.375, 22.5, 28.75, 24.375]), 90.0, Field(Face[Row][3:4]: [21.875]))
operators: ['heat_step']
TypeError: 'with_exchange' is not a valid program definition: uses the collective 'exchange'.


## Distributions

A `Distribution` is configuration, never in the source of a program: `distribute(row, owner, support=..., hops=...)` gives each rank the points `owner` assigns to it, and a halo of `hops` steps over the relations of `support`; `derive_distribution(row, owner, programs=[...])` gives it a halo of exactly what the reads of the programs' operators need. `check_coverage` raises a `ValueError` if a halo is too small.

In depth: [13. Global programs](13_global_programs.ipynb).

In [6]:
from implementation import check_coverage, derive_distribution, distribute

# each edge goes with its right cell, e4 with c3: rank 0 owns c0, c1, e0, e1, rank 1 the rest
owner = {Cell: [0, 0, 1, 1], Edge: [0, 0, 1, 1, 1]}
halves = distribute(row, owner, support=(C2E, E2C), hops=2)  # given: 2 hops over C2E and E2C
derived = derive_distribution(row, owner, programs=[simulate])  # from the reads of heat_step


def indices(distribution):  # the global indices of the local cells and edges of each rank
    return [(lm.global_index[Cell], lm.global_index[Edge]) for lm in distribution.local_meshes]


# owned first, then the halo: c2 reads e2 and e3, and e2 reads c1, so rank 1 needs c1
print("given:  ", indices(halves))
print("derived:", indices(derived), "| mesh_id:", derived.local_meshes[0].mesh_id)
try:
    check_coverage(simulate.operators, distribute(row, owner, support=(C2E, E2C), hops=1))
except ValueError as error:  # 1 hop: rank 0 reaches e2 from c1, not c2 from e2
    print("ValueError:", error)

given:   [((0, 1, 2), (0, 1, 2)), ((2, 3, 1), (2, 3, 4, 1))]
derived: [((0, 1, 2), (0, 1, 2)), ((2, 3, 1), (2, 3, 4))] | mesh_id: Row-0
ValueError: The distribution of 'Row-0' does not cover the reads of 'heat_step': rank 0 needs Face[Row][2:3] (distribute with more `hops`, or derive the distribution).


## The lowering

`simulate.lower(distribution)` checks the coverage and returns a `LoweredProgram`. Called with mesh fields, it gives each rank its owned values, runs the body on every rank as a local program, with an exchange before each operator call that reads a field through a relation (unless the field was already exchanged), and gathers the results; `lowered.log` lists the actions of rank 0. The simulation condition: the lowered results are the direct ones, bitwise, on 1, 2 and 3 ranks.

In depth: [13. Global programs](13_global_programs.ipynb).

In [7]:
def same(a, b):  # bitwise the same results: fields (domains, values, mesh_id), tuples, scalars
    if isinstance(a, tuple):
        return isinstance(b, tuple) and len(a) == len(b) and all(map(same, a, b))
    if isinstance(a, Field):
        return (a.domain, a.data.tolist(), a.mesh.mesh_id) == (
            b.domain,
            b.data.tolist(),
            b.mesh.mesh_id,
        )
    return type(a) is type(b) and a == b


lowered = simulate.lower(halves)  # checks the coverage first
results = lowered(*args)
print("lowered:", results)
# t is exchanged before each step, edge_sign once (it does not change); cell_sign is read
# on the owned cells only; then the step on East, and the global sum
print("log on halves:", *lowered.log, sep="\n  ")
single = distribute(row, {Cell: [0, 0, 0, 0], Edge: [0, 0, 0, 0, 0]})
thirds = distribute(row, {Cell: [0, 1, 1, 2], Edge: [0, 1, 1, 2, 2]}, support=(C2E, E2C), hops=2)
equal = [same(simulate.lower(d)(*args), direct) for d in (single, halves, thirds, derived)]
print("lowered == direct on single, halves, thirds, derived:", equal)

lowered: (Field(Face[Row][0:4]: [14.375, 22.5, 28.75, 24.375]), 90.0, Field(Face[Row][3:4]: [21.875]))
log on halves:
  exchange 't' for 'heat_step'
  exchange 'edge_sign' for 'heat_step'
  compute 'heat_step' on the owned points
  exchange 't' for 'heat_step'
  compute 'heat_step' on the owned points
  exchange 't' for 'heat_step'
  compute 'heat_step' on the owned points
  global_sum over 'Face[Row]'
lowered == direct on single, halves, thirds, derived: [True, True, True, True]


## Backends and the compile key

`program(backend=compiled)(simulate.function)` runs the same function with its operators on a backend. They are built once per operator and mesh signature: the direct run, the lowered runs and a longer row of 6 cells, with the same relations and regions, share one compiled `heat_step`.

In depth: [13. Global programs](13_global_programs.ipynb).

In [8]:
from implementation import compiled

compiled_simulate = program(backend=compiled)(simulate.function)  # the same function, compiled
long_row = LogicalMesh(  # 6 cells: other tables and extents, the same signature
    Row, row_tables(6), extents={Cell: 6, Edge: 7}, regions={West: (0,), East: (5,)}
)
print("same signature:", long_row.signature == row.signature)
long_args = (long_row.full(Cell, fill_value=t_out), t_out, k, *signs(long_row), 2)
with contextlib.redirect_stdout(io.StringIO()) as printed:  # silence the backend
    on_row = compiled_simulate(*args)
    on_halves = compiled_simulate.lower(halves)(*args)
    on_long_row = compiled_simulate(*long_args)
print([line for line in printed.getvalue().splitlines() if "compiling" in line])  # once
print("compiled == embedded:", same(on_row, direct), same(on_halves, direct))
# at t_out everywhere, nothing flows: 6 * 10 = 60 in total, and the East cell stays at 10
print("long row:", on_long_row)

same signature: True
['[compiled] compiling heat_step']
compiled == embedded: True True
long row: (Field(Face[Row][0:6]: [10.0, 10.0, 10.0, 10.0, 10.0, 10.0]), 60.0, Field(Face[Row][5:6]: [10.0]))


## L1 next to L2

The same simulation as an L1 local program, on the local meshes of the distribution `halves`: the decomposition is in its source. In the L2 program, it moved to configuration:

| | L1: `@local_program`, run by `run_local` | L2: `@program()`, lowered by `lower` |
|---|---|---|
| The mesh | a `LocalMesh` per rank, from `decompose`, in `with` blocks | a `LogicalMesh`, carried by its fields |
| Results on | `heat_step[Owned(Cell)](...)` | the whole mesh, `heat_step(...)`, or a region, `heat_step[region(East)](...)` |
| Halos | `exchange(t)` before each step, by hand | placed by the lowering |
| Sums | `global_sum(t, domain=Owned(Cell))` | `sum_over(t, axis=Cell)` |
| Decomposition | in the source | a `Distribution`, given to `lower` |
| Inputs and results | `scatter`, `run_local`, `gather_owned` | mesh fields in, mesh fields out |

In depth: [10. Local programs](10_local_programs.ipynb), [13. Global programs](13_global_programs.ipynb).

In [9]:
from concepts import Owned
from implementation import gather_owned, global_sum, local_program, run_local, scatter


@local_program
def simulate_local(t, t_out, k, edge_sign, cell_sign, steps):  # L1: per rank, by hand
    edge_sign = exchange(edge_sign)
    for _ in range(steps):
        t = exchange(t)
        t = heat_step[Owned(Cell)](t, t_out, k, edge_sign, cell_sign)
    return t, global_sum(t, domain=Owned(Cell))


ranks = halves.local_meshes  # the `LocalMesh`es of the distribution, as from `decompose`
per_rank = [
    (scatter(t, r), t_out, k, scatter(edge_sign, r), scatter(cell_sign, r), 2) for r in ranks
]
local_results = run_local(simulate_local, ranks, per_rank)
l1 = gather_owned([r[0] for r in local_results], ranks), local_results[0][1]
print("L1, local program:  ", l1[0], l1[1])
print("L2, global program: ", results[0], results[1])
print("the same values:", l1[0].data.tolist() == results[0].data.tolist() and l1[1] == results[1])

L1, local program:   Field(Face[Row][0:4]: [14.375, 22.5, 28.75, 24.375]) 90.0
L2, global program:  Field(Face[Row][0:4]: [14.375, 22.5, 28.75, 24.375]) 90.0
the same values: True


## Cheat sheet

| I want to... | Write... | In depth |
|---|---|---|
| declare a schema and its locations | `class Row(Schema): ...`, `Cell = Face[Row]`, `Edge = concepts.Edge[Row]` | [12](12_logical_meshes.ipynb) |
| declare regions | `class East(Region[Cell]): ...` | [12](12_logical_meshes.ipynb) |
| build the mesh | `LogicalMesh(Row, tables, extents=..., regions=..., geometry=...)` | [12](12_logical_meshes.ipynb) |
| make fields of the mesh | `row.field(Cell, data=...)`, `row.zeros(Edge)`, `row.full(Cell, fill_value=...)`, `mesh_field(row, domain, data)` | [12](12_logical_meshes.ipynb) |
| get the points of a region | `row.region(East)`; in a program, `region(East)` | [12](12_logical_meshes.ipynb), [13](13_global_programs.ipynb) |
| run L1 operators on the mesh | call them with mesh fields: no `with` block | [12](12_logical_meshes.ipynb) |
| write a program | `@program()` on a function calling operators | [13](13_global_programs.ipynb) |
| sum over the whole mesh | `sum_over(t, axis=Cell)`, in a program | [13](13_global_programs.ipynb) |
| distribute the mesh | `distribute(row, owner, support=..., hops=...)`, `derive_distribution(row, owner, programs=[...])` | [13](13_global_programs.ipynb) |
| check a halo | `check_coverage(simulate.operators, distribution)` | [13](13_global_programs.ipynb) |
| run on the ranks | `simulate.lower(distribution)(...)`, then `lowered.log` | [13](13_global_programs.ipynb) |
| run on a backend | `program(backend=compiled)(simulate.function)` | [13](13_global_programs.ipynb) |
| share compiled operators | meshes with the same `signature` | [12](12_logical_meshes.ipynb), [13](13_global_programs.ipynb) |

Back: [Quickstart L1: local semantics](quickstart_l1.ipynb), the L1 core of this example.